# Advanced Pipelines

## Prepare

In [1]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("ticks")
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["figure.figsize"] = [10, 6]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = False
plt.rcParams["grid.color"] = "white"

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    Normalizer,
    QuantileTransformer,
    MinMaxScaler,
    KBinsDiscretizer,
    PolynomialFeatures,
    FunctionTransformer,
)
from sklearn.impute import SimpleImputer

In [3]:
from sklearn.model_selection import cross_val_score

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [5]:
DATA_PATH = os.path.join("..", "data", "titanic.csv")

In [6]:
np.random.randn(42)
random.seed(42)

## Import

In [7]:
df_raw = pd.read_csv(DATA_PATH)

In [8]:
train_df, test_df = train_test_split(df_raw, random_state=42)

## Transform

In [11]:
target_col = "survived"
num_cols = ["sibsp", "parch", "age", "fare"]
cat_cols = ["sex", "embarked", "pclass"]

In [12]:
def split_x_y(df):
    y = np.array(df[target_col])
    return df, y

In [13]:
X_train, y_train = split_x_y(train_df)

In [14]:
num_pipeline = Pipeline(
    [
        ("simple_imp", SimpleImputer()),
        ("pol", PolynomialFeatures()),
    ]
)

In [15]:
cat_pipeline = Pipeline(
    [
        ("one_hot", OneHotEncoder()),
    ]
)

In [16]:
log_transform = FunctionTransformer(np.log1p, validate=True)

In [17]:
age_pipeline = Pipeline(
    [
        ("imp", SimpleImputer()),
        ("qua", QuantileTransformer(n_quantiles=100)),
    ]
)

In [18]:
transform_pipeline = ColumnTransformer(
    [
        ("log", log_transform, ["fare"]),
        #    ("age", age_pipeline, ["age"]),
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols),
    ]
)

In [19]:
transform_pipeline.fit_transform(train_df).shape

(668, 25)

## Analysis

In [20]:
regression_pipeline = Pipeline(
    [
        ("transform", transform_pipeline),
        ("logit", LogisticRegression(solver="liblinear", random_state=42)),
    ]
)

In [21]:
model = regression_pipeline.fit(train_df, y_train)

In [22]:
model.score(train_df, y_train)

0.812874251497006

In [23]:
c_scores = cross_val_score(
    regression_pipeline, train_df, y_train, scoring="accuracy", cv=10
)
c_scores

array([0.85074627, 0.71641791, 0.85074627, 0.89552239, 0.82089552,
       0.74626866, 0.70149254, 0.79104478, 0.75757576, 0.93939394])

In [24]:
c_scores.mean()

0.8070104025327905

## Review

In [25]:
X_test, y_test = split_x_y(test_df)

In [26]:
model.score(X_test, y_test)

0.8071748878923767